<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Ollama를 통한 Llama 3 모델을 사용하여 로컬에서 지시 응답 평가하기

- 이 노트북은 ollama를 통해 80억 파라미터의 Llama 3 모델을 사용하여 지시 미세조정된 LLM의 응답을 평가합니다. 평가는 생성된 모델 응답이 포함된 JSON 형식의 데이터셋을 기반으로 합니다. 예를 들어:



```python
{
    "instruction": "What is the atomic number of helium?",
    "input": "",
    "output": "The atomic number of helium is 2.",               # <-- 테스트 세트에서 주어진 목표 답변
    "model 1 response": "\nThe atomic number of helium is 2.0.", # <-- LLM의 응답
    "model 2 response": "\nThe atomic number of helium is 3."    # <-- 두 번째 LLM의 응답
},
```

- 이 코드는 GPU가 필요하지 않으며 노트북에서 실행됩니다 (M3 MacBook Air에서 테스트됨)

In [1]:
from importlib.metadata import version

pkgs = ["tqdm",    # 진행률 표시줄
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

tqdm version: 4.66.4


## Ollama 설치 및 Llama 3 다운로드

- Ollama는 LLM을 효율적으로 실행하는 애플리케이션입니다
- 이는 효율성을 극대화하기 위해 순수 C/C++로 LLM을 구현한 [llama.cpp](https://github.com/ggerganov/llama.cpp)를 감싸는 래퍼입니다
- 이것은 텍스트 생성(추론)을 위한 LLM 사용 도구이며, LLM 학습이나 미세조정을 위한 것이 아님에 주목하세요
- 아래 코드를 실행하기 전에 [https://ollama.com](https://ollama.com)을 방문하여 지침을 따라 ollama를 설치하세요 (예: "Download" 버튼을 클릭하고 운영 체제용 ollama 애플리케이션을 다운로드)

- macOS 및 Windows 사용자의 경우 다운로드한 ollama 애플리케이션을 클릭하세요. 명령줄 사용을 설치하라는 메시지가 표시되면 "yes"라고 답하세요
- Linux 사용자는 ollama 웹사이트에 제공된 설치 명령어를 사용할 수 있습니다

- 일반적으로 명령줄에서 ollama를 사용하기 전에 ollama 애플리케이션을 시작하거나 별도의 터미널에서 `ollama serve`를 실행해야 합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- ollama 애플리케이션이나 `ollama serve`가 실행 중인 상태에서 다른 터미널의 명령줄에서 다음 명령을 실행하여 80억 파라미터의 Llama 3 모델을 시도해보세요 (이 명령을 처음 실행할 때 4.7GB의 저장 공간을 차지하는 모델이 자동으로 다운로드됩니다)

```bash
# 8B 모델
ollama run llama3
```


출력은 다음과 같습니다:

```
$ ollama run llama3
pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                         
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                         
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                         
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                         
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
removing any unused layers 
success 
```

- `llama3`는 지시 미세조정된 80억 파라미터 Llama 3 모델을 가리킵니다

- 또는 머신이 지원하는 경우 `llama3`를 `llama3:70b`로 바꾸어 더 큰 700억 파라미터 Llama 3 모델을 사용할 수도 있습니다

- 다운로드가 완료된 후 모델과 채팅할 수 있는 명령줄 프롬프트가 나타납니다

- "What do llamas eat?"와 같은 프롬프트를 시도해보세요. 다음과 비슷한 출력을 반환해야 합니다:

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- `/bye` 입력을 사용하여 이 세션을 종료할 수 있습니다

## Ollama의 REST API 사용

- 이제 모델과 상호작용하는 대안적인 방법은 다음 함수를 통해 Python의 REST API를 사용하는 것입니다
- 이 노트북의 다음 셀들을 실행하기 전에 위에서 설명한 대로 ollama가 여전히 실행 중인지 확인하세요:
  - 터미널에서 `ollama serve`
  - ollama 애플리케이션
- 다음으로 모델을 쿼리하기 위해 다음 코드 셀을 실행하세요

- 먼저 간단한 예제로 API를 시도해서 의도한 대로 작동하는지 확인해보겠습니다:

In [2]:
import urllib.request
import json


def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    # 데이터 페이로드를 딕셔너리로 생성
    data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {     # 결정론적 응답을 위해 아래 설정이 필요합니다
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # 딕셔너리를 JSON 형식의 문자열로 변환하고 바이트로 인코딩
    payload = json.dumps(data).encode("utf-8")

    # 요청 객체를 생성하고 메서드를 POST로 설정하며 필요한 헤더 추가
    request = urllib.request.Request(url, data=payload, method="POST")
    request.add_header("Content-Type", "application/json")

    # 요청을 보내고 응답을 캡처
    response_data = ""
    with urllib.request.urlopen(request) as response:
        # 응답을 읽고 디코딩
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data


result = query_model("What do Llamas eat?")
print(result)

Llamas are herbivores, which means they primarily feed on plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and even weeds.
2. Hay: High-quality hay, such as alfalfa or timothy hay, is a staple in a llama's diet. They enjoy the sweet taste and texture of fresh hay.
3. Grains: Llamas may receive grains like oats, barley, or corn as part of their daily ration. However, it's essential to provide these grains in moderation, as they can be high in calories.
4. Fruits and vegetables: Llamas enjoy a variety of fruits and veggies, such as apples, carrots, sweet potatoes, and leafy greens like kale or spinach.
5. Minerals: Llamas require access to mineral supplements, which help maintain their overall health and well-being.

In the wild, llamas might also eat:

1. Leaves: They'll munch on leaves from trees and shrubs, including plants like willow, alder, and birch.
2. Bark: In some cases, ll

## JSON 항목 로드

- 이제 데이터 평가 부분으로 넘어가겠습니다
- 여기서는 테스트 데이터셋과 모델 응답을 다음과 같이 로드할 수 있는 JSON 파일로 저장했다고 가정합니다:

In [3]:
json_file = "eval-example-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

Number of entries: 100


- 이 파일의 구조는 다음과 같습니다. 여기서 테스트 데이터셋에서 주어진 응답(`'output'`)과 두 개의 서로 다른 모델의 응답(`'model 1 response'` 및 `'model 2 response'`)이 있습니다:

In [4]:
json_data[0]

{'instruction': 'Calculate the hypotenuse of a right triangle with legs of 6 cm and 8 cm.',
 'input': '',
 'output': 'The hypotenuse of the triangle is 10 cm.',
 'model 1 response': '\nThe hypotenuse of the triangle is 3 cm.',
 'model 2 response': '\nThe hypotenuse of the triangle is 12 cm.'}

- 아래는 나중에 시각화 목적을 위해 입력을 형식화하는 작은 유틸리티 함수입니다:

In [5]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- 이제 ollama API를 시도해서 모델 응답을 비교해보겠습니다 (시각적 비교를 위해 처음 5개의 응답만 평가합니다):

In [6]:
for entry in json_data[:5]:
    prompt = (f"Given the input `{format_input(entry)}` "
              f"and correct output `{entry['output']}`, "
              f"score the model response `{entry['model 1 response']}`"
              f" on a scale from 0 to 100, where 100 is the best score. "
              )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model 1 response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


Dataset response:
>> The hypotenuse of the triangle is 10 cm.

Model response:
>> 
The hypotenuse of the triangle is 3 cm.

Score:
>> I'd score this response as 0 out of 100.

The correct answer is "The hypotenuse of the triangle is 10 cm.", not "3 cm.". The model failed to accurately calculate the length of the hypotenuse, which is a fundamental concept in geometry and trigonometry.

-------------------------

Dataset response:
>> 1. Squirrel
2. Eagle
3. Tiger

Model response:
>> 
1. Squirrel
2. Tiger
3. Eagle
4. Cobra
5. Tiger
6. Cobra

Score:
>> I'd rate this model response as 60 out of 100.

Here's why:

* The model correctly identifies two animals that are active during the day: Squirrel and Eagle.
* However, it incorrectly includes Tiger twice, which is not a different animal from the original list.
* Cobra is also an incorrect answer, as it is typically nocturnal or crepuscular (active at twilight).
* The response does not meet the instruction to provide three different animals

- 응답이 매우 장황함에 주목하세요. 어느 모델이 더 나은지 정량화하기 위해서는 점수만 반환하고 싶습니다:

In [7]:
from tqdm import tqdm


def generate_model_scores(json_data, json_key):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt)
        try:
            scores.append(int(score))
        except ValueError:
            continue

    return scores

- 이제 이 평가를 전체 데이터셋에 적용하고 각 모델의 평균 점수를 계산해보겠습니다 (M3 MacBook Air 노트북에서 모델당 약 1분 소요)
- ollama는 (이 글을 쓰는 시점에서) 운영 체제 간에 완전히 결정론적이지 않으므로 아래 표시된 것과 약간 다른 숫자가 나올 수 있습니다

In [8]:
from pathlib import Path

for model in ("model 1 response", "model 2 response"):

    scores = generate_model_scores(json_data, model)
    print(f"\n{model}")
    print(f"Number of scores: {len(scores)} of {len(json_data)}")
    print(f"Average score: {sum(scores)/len(scores):.2f}\n")

    # 선택적으로 점수를 저장
    save_path = Path("scores") / f"llama3-8b-{model.replace(' ', '-')}.json"
    with open(save_path, "w") as file:
        json.dump(scores, file)

Scoring entries: 100%|████████████████████████| 100/100 [01:02<00:00,  1.59it/s]



model 1 response
Number of scores: 100 of 100
Average score: 78.48



Scoring entries: 100%|████████████████████████| 100/100 [01:10<00:00,  1.42it/s]



model 2 response
Number of scores: 99 of 100
Average score: 64.98



- 위의 평가를 바탕으로 첫 번째 모델이 두 번째 모델보다 더 좋다고 말할 수 있습니다